In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
import seaborn as sns
import torchmetrics.functional as tmf
import torchmetrics
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, GradientAccumulationScheduler
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from torchvision import models
from pytorch_lightning.loggers import WandbLogger
import torchio as tio
import random
import gc
import os
import monai
import monai.transforms as monai_transforms

import numpy as np
import torch
from torch.utils.data import Dataset
import torchio as tio
import monai.transforms as monai_transforms
from torchvision import transforms

import numpy as np
import torch
from scipy.ndimage import distance_transform_edt

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def set_random_seed(seed=42):
    # seed setting
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    print(f"Random seed set to {seed}")

set_random_seed(seed=42)

seed = 42
pl.seed_everything(seed = 42)


# Distance Transform

In [ ]:
'''
csv_path = "/home/zobia/Boston_Datasets/MRI-data-20250805T065054Z-1-007/MRI-data/csv_files/NACC_train_80%.csv"

df = pd.read_csv(csv_path)

distance_dir = "/home/zobia/Boston_Datasets/NACC_distance_maps"
os.makedirs(distance_dir, exist_ok=True)

for path in tqdm(df["full_path"]):

    mri = np.load(path)

    mask = mri > 0
    dist = distance_transform_edt(mask)

    dist = dist / (dist.max() + 1e-8)

    filename = os.path.basename(path).replace(".npy", "_distance.npy")
    save_path = os.path.join(distance_dir, filename)

    np.save(save_path, dist)

print("Distance transforms created.")
'''

# Dataset Processing

In [ ]:
batch_size = 2

def worker_init_fn(worker_id):
    np.random.seed(seed + worker_id)
    random.seed(seed + worker_id)

# Function to normalize the data
def minmax_normalize(data):
    return (data - data.min()) / (data.max() - data.min() + 0.0005)


In [ ]:
class DataLoading(Dataset):
    def __init__(self, file_paths, labels, distance_dir=None, transform=True, augmentation=True, use_distances=False):
        self.file_paths = file_paths
        self.labels = labels
        self.distance_dir = distance_dir
        self.transform = transform
        self.use_distances = use_distances  # Toggle for loading distance transforms

        # Define transformations
        if transform and augmentation:
            self.transforms = tio.Compose([
                transforms.RandomApply(
                    [monai_transforms.RandSpatialCrop(
                        roi_size=(182//4, 218//4, 182//4),
                        random_center=True,
                        random_size=True
                    ),
                    monai_transforms.Resize(
                        spatial_size=(182, 218, 182)
                    )], p=0.5
                ),
                tio.RandomGamma(p=0.5),
                tio.RandomBiasField(p=0.25),
                tio.Lambda(minmax_normalize)
            ])
        elif transform and not augmentation:
            self.transforms = tio.Compose([
                tio.RandomGamma(p=0.5),
                tio.RandomBiasField(p=0.25),
                tio.Lambda(minmax_normalize)
            ])
        else:
            self.transforms = tio.Compose([
                tio.Lambda(minmax_normalize)
            ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]

        # Load MRI data
        data = np.load(file_path)
        data = np.expand_dims(data, axis=0)  # Add channel dimension
        data_tensor = torch.tensor(data, dtype=torch.float32)

        # Optionally load corresponding distance transform
        distance_tensor = None
        if self.use_distances and self.distance_dir:
            file_name = os.path.basename(file_path).replace('.npy', '_distance.npy')
            distance_path = os.path.join(self.distance_dir, file_name)
            distance_transform = np.load(distance_path)
            distance_tensor = torch.tensor(distance_transform, dtype=torch.float32)

        # Apply transformations (to MRI data only)
        if self.transform:
            data_tensor = self.transforms(data_tensor)

        # Convert label to PyTorch tensor
        label_tensor = torch.tensor(label, dtype=torch.float)

        if self.use_distances:
            return data_tensor, label_tensor, distance_tensor
        return data_tensor, label_tensor


In [ ]:
class DataModule(pl.LightningDataModule):
    def __init__(self, train_dataset, val_dataset, test_dataset, batch_size, distributed_sampler=False):
        super().__init__()
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.test_dataset = test_dataset
        self.batch_size = batch_size
        self.distributed_sampler = distributed_sampler

    def _get_data_loader(self, dataset, shuffle):
        sampler = DistributedSampler(dataset) if self.distributed_sampler else None

        def collate_fn(batch):
            if len(batch[0]) == 3:  # If distance transforms are included
                data_batch, label_batch, distance_batch = zip(*batch)
        
                # Process MRI data
                resized_data_batch = []
                for data in data_batch:
                    if data.dim() == 4:  # (C, X, Y, Z)
                        data = data.unsqueeze(0)  # Add batch dimension: (1, C, X, Y, Z)
                        data = F.interpolate(data, size=(176, 208, 176), mode='trilinear', align_corners=False)
                        data = data.squeeze(0)  # Remove batch dimension: (C, X', Y', Z')
                    elif data.dim() == 5:  # (B, C, X, Y, Z)
                        data = F.interpolate(data, size=(176, 208, 176), mode='trilinear', align_corners=False)
                    resized_data_batch.append(data)
        
                resized_data_batch = torch.stack(resized_data_batch)
                label_batch = torch.stack(label_batch)
                distance_batch = torch.stack(distance_batch)  # Stack distance transforms
        
                return resized_data_batch, label_batch, distance_batch
        
            else:  # Without distance transforms
                data_batch, label_batch = zip(*batch)
        
                # Process MRI data
                resized_data_batch = []
                for data in data_batch:
                    if data.dim() == 4:  # (C, X, Y, Z)
                        data = data.unsqueeze(0)  # Add batch dimension: (1, C, X, Y, Z)
                        data = F.interpolate(data, size=(176, 208, 176), mode='trilinear', align_corners=False)
                        data = data.squeeze(0)  # Remove batch dimension: (C, X', Y', Z')
                    elif data.dim() == 5:  # (B, C, X, Y, Z)
                        data = F.interpolate(data, size=(176, 208, 176), mode='trilinear', align_corners=False)
                    resized_data_batch.append(data)
        
                resized_data_batch = torch.stack(resized_data_batch)
                label_batch = torch.stack(label_batch)
        
                return resized_data_batch, label_batch

        return DataLoader(dataset, batch_size=self.batch_size, shuffle=shuffle, sampler=sampler, num_workers=15, collate_fn=collate_fn,worker_init_fn=worker_init_fn)

    def train_dataloader(self):
        return self._get_data_loader(self.train_dataset, shuffle=True)

    def val_dataloader(self):
        return self._get_data_loader(self.val_dataset, shuffle=False)

    def test_dataloader(self):
        return self._get_data_loader(self.test_dataset, shuffle=False)

In [ ]:

# Load NACC_60%.csv for training
nacc_train_data = pd.read_csv('/home/zobia/Boston_Datasets/MRI-data-20250805T065054Z-1-007/MRI-data/csv_files/ADNI_ALL_test_100%.csv')
X_nacc_train = nacc_train_data['full_path'].values
y_nacc_train = nacc_train_data[['NC', 'MCI', 'AD']].values

# Load NACC_20%.csv for validation
nacc_val_data = pd.read_csv('//home/zobia/Boston_Datasets/MRI-data-20250805T065054Z-1-007/MRI-data/csv_files/ADNI_ALL_test_100%.csv')
X_nacc_val = nacc_val_data['full_path'].values
y_nacc_val = nacc_val_data[['NC', 'MCI', 'AD']].values

# Load ADNI_85%.csv for testing
adni_test_data = pd.read_csv('/home/zobia/Boston_Datasets/MRI-data-20250805T065054Z-1-007/MRI-data/csv_files/AIBL_test_100%.csv')
X_adni_test = adni_test_data['full_path'].values
y_adni_test = adni_test_data[['NC', 'MCI', 'AD']].values

# Total data set lengths
total_samples_train = len(X_nacc_train)
print(f"Total training data length: {total_samples_train}")
print(f"Total validation data length: {len(X_nacc_val)}")
print(f"Total testing data length: {len(X_adni_test)}")

# Class counts for the training dataset
nc_count_train = y_nacc_train[:, 0].sum()
mci_count_train = y_nacc_train[:, 1].sum()
ad_count_train = y_nacc_train[:, 2].sum()
print(f"Training dataset class counts: NC={nc_count_train}, MCI={mci_count_train}, AD={ad_count_train}")

# Class counts for validation dataset
nc_count_val = y_nacc_val[:, 0].sum()
mci_count_val = y_nacc_val[:, 1].sum()
ad_count_val = y_nacc_val[:, 2].sum()
print(f"Validation dataset class counts: NC={nc_count_val}, MCI={mci_count_val}, AD={ad_count_val}")

# Class counts for testing dataset
nc_count_test = y_adni_test[:, 0].sum()
mci_count_test = y_adni_test[:, 1].sum()
ad_count_test = y_adni_test[:, 2].sum()
print(f"Testing dataset class counts: NC={nc_count_test}, MCI={mci_count_test}, AD={ad_count_test}")

# Now create instances of the DataLoading for each split
train_dataset = DataLoading(
    X_nacc_train, 
    y_nacc_train, 
    distance_dir="/home/zobia/NACC/distance_transforms/", 
    transform=True, 
    augmentation=False, 
    use_distances=True  # Enable distance transforms for training
)

# Validation dataset (without distance transforms)
val_dataset = DataLoading(
    X_nacc_val, 
    y_nacc_val, 
    transform=False, 
    augmentation=False, 
    use_distances=False  # No distance transforms for validation
)

# Test dataset (without distance transforms)
test_dataset = DataLoading(
    X_adni_test, 
    y_adni_test, 
    transform=False, 
    augmentation=False, 
    use_distances=False  # No distance transforms for testing
)

In [ ]:
# Calculate class weights for training dataset
class_weights = {
    0: total_samples_train / nc_count_train,
    1: total_samples_train / mci_count_train,
    2: total_samples_train / ad_count_train
}
print(f"Class weights for training dataset: {class_weights}")
# weights_tensor = torch.tensor([class_weights[i] for i in range(len(class_weights))], dtype=torch.float)

# UNET 3D

In [ ]:
class ContBatchNorm3d(nn.modules.batchnorm._BatchNorm):
    def _check_input_dim(self, input):

        if input.dim() != 5:
            raise ValueError('expected 5D input (got {}D input)'.format(input.dim()))
        #super(ContBatchNorm3d, self)._check_input_dim(input)

    def forward(self, input):
        self._check_input_dim(input)
        return F.batch_norm(
            input, self.running_mean, self.running_var, self.weight, self.bias,
            True, self.momentum, self.eps)

class LUConv(nn.Module):
    def __init__(self, in_chan, out_chan, act):
        super(LUConv, self).__init__()
        self.conv1 = nn.Conv3d(in_chan, out_chan, kernel_size=3, padding=1)
        self.bn1 = ContBatchNorm3d(out_chan)

        if act == 'relu':
            self.activation = nn.ReLU(out_chan)
        elif act == 'prelu':
            self.activation = nn.PReLU(out_chan)
        elif act == 'elu':
            self.activation = nn.ELU(inplace=True)
        else:
            raise

    def forward(self, x):
        out = self.activation(self.bn1(self.conv1(x)))
        return out


def _make_nConv(in_channel, depth, act, double_chnnel=False):
    if double_chnnel:
        layer1 = LUConv(in_channel, 32 * (2 ** (depth+1)),act)
        layer2 = LUConv(32 * (2 ** (depth+1)), 32 * (2 ** (depth+1)),act)
    else:
        layer1 = LUConv(in_channel, 32*(2**depth),act)
        layer2 = LUConv(32*(2**depth), 32*(2**depth)*2,act)

    return nn.Sequential(layer1,layer2)

class DownTransition(nn.Module):
    def __init__(self, in_channel,depth, act):
        super(DownTransition, self).__init__()
        self.ops = _make_nConv(in_channel, depth,act)
        self.maxpool = nn.MaxPool3d(2)
        self.current_depth = depth

    def forward(self, x):
        if self.current_depth == 3:
            out = self.ops(x)
            out_before_pool = out
        else:
            out_before_pool = self.ops(x)
            out = self.maxpool(out_before_pool)
        return out, out_before_pool

class UpTransition(nn.Module):
    def __init__(self, inChans, outChans, depth,act):
        super(UpTransition, self).__init__()
        self.depth = depth
        self.up_conv = nn.ConvTranspose3d(inChans, outChans, kernel_size=2, stride=2)
        self.ops = _make_nConv(inChans+ outChans//2,depth, act, double_chnnel=True)

    def forward(self, x, skip_x):
        out_up_conv = self.up_conv(x)
        concat = torch.cat((out_up_conv,skip_x),1)
        out = self.ops(concat)
        return out

class OutputTransition(nn.Module):
    def __init__(self, inChans, n_labels):

        super(OutputTransition, self).__init__()
        self.final_conv = nn.Conv3d(inChans, n_labels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.sigmoid(self.final_conv(x))
        return out

class UNet3D(nn.Module):

    def __init__(self, n_class=1, act='relu'):
        super(UNet3D, self).__init__()

        self.down_tr64 = DownTransition(1,0,act)
        self.down_tr128 = DownTransition(64,1,act)
        self.down_tr256 = DownTransition(128,2,act)
        self.down_tr512 = DownTransition(256,3,act)

        self.up_tr256 = UpTransition(512, 512,2,act)
        self.up_tr128 = UpTransition(256,256, 1,act)
        self.up_tr64 = UpTransition(128,128,0,act)
        self.out_tr = OutputTransition(64, n_class)

    def forward(self, x):
        self.out64, self.skip_out64 = self.down_tr64(x)
        self.out128,self.skip_out128 = self.down_tr128(self.out64)
        self.out256,self.skip_out256 = self.down_tr256(self.out128)
        self.out512,self.skip_out512 = self.down_tr512(self.out256)

        #self.out_up_256 = self.up_tr256(self.out512,self.skip_out256)
        #self.out_up_128 = self.up_tr128(self.out_up_256, self.skip_out128)
        #self.out_up_64 = self.up_tr64(self.out_up_128, self.skip_out64)
        #self.out = self.out_tr(self.out_up_64)

        return self.out512
    
base_model=UNet3D()

In [ ]:
class TargetNet(nn.Module):
    def __init__(self, base_model, n_classes=3):
        super(TargetNet, self).__init__()
        self.base_model = base_model
        self.dense_1 = nn.Linear(512, 1024, bias=True)
        self.dense_2 = nn.Linear(1024, n_classes, bias=True)

    def forward(self, x):
        
        #self.base_model(x)
        #self.base_out = self.base_model.out512
        #self.out_glb_avg_pool = F.avg_pool3d(self.base_out, kernel_size=self.base_out.size()[2:]).view(self.base_out.size()[0], -1)

        base_out = self.base_model(x)  # Capture the return value from base_model directly
        self.out_glb_avg_pool = F.avg_pool3d(base_out, kernel_size=base_out.size()[2:]).view(base_out.size()[0], -1)
        self.linear_out = self.dense_1(self.out_glb_avg_pool)
        probabilities = self.dense_2(F.relu(self.linear_out))
        #probabilities = F.softmax(final_out, dim=1)

        return probabilities

# Lightening module and utilities

In [ ]:
def compute_metrics(y_true, y_pred):
    num_classes = 3
    overall_accuracy = tmf.accuracy(y_pred, y_true, num_classes=num_classes, task='multiclass')
    #weighted_acc = tmf.accuracy(y_pred, y_true, average='weighted', task='multiclass', num_classes=num_classes)
    #weighted_precision = tmf.precision(y_pred, y_true, average='weighted', task='multiclass', num_classes=num_classes)
    #weighted_recall = tmf.recall(y_pred, y_true, average='weighted', task='multiclass', num_classes=num_classes)
    f1_macro = tmf.f1_score(y_pred, y_true, average='macro', task='multiclass', num_classes=num_classes)
    #f1_weighted = tmf.f1_score(y_pred, y_true, average='weighted', task='multiclass', num_classes=num_classes)
    macro_precision = tmf.precision(y_pred, y_true, average='macro', task='multiclass', num_classes=num_classes)
    macro_recall = tmf.recall(y_pred, y_true, average='macro', task='multiclass', num_classes=num_classes)

    # Prepare the result dictionary
    return {
        'overall_accuracy': overall_accuracy.item(),
        'macro_precision': macro_precision.item(),
        'macro_recall': macro_recall.item(),
        'f1_macro': f1_macro.item(),
    }

In [ ]:
class SoftCrossEntropyLoss(nn.Module):
    def __init__(self, weights):
        """
        Implements Soft Cross-Entropy Loss with class weights.

        Args:
        - weights (torch.Tensor): Class weights tensor of shape (num_classes,).
        """
        super(SoftCrossEntropyLoss, self).__init__()
        self.weights = weights

    def forward(self, y_hat, y):
        """
        Compute soft cross-entropy loss with class weights.

        Args:
        - y_hat (torch.Tensor): Logits from the model, shape (batch_size, num_classes).
        - y (torch.Tensor): Soft or one-hot encoded labels, shape (batch_size, num_classes).

        Returns:
        - loss (torch.Tensor): Scalar loss value.
        """
        # Log-softmax on model logits
        p = F.log_softmax(y_hat, dim=1)

        # Apply weights to the targets
        w_labels = self.weights.to(y_hat.device) * y

        # Compute weighted soft cross-entropy loss
        loss = -(w_labels * p).sum() / w_labels.sum()
        return loss

In [ ]:
class LightningModule(pl.LightningModule):
    def __init__(self, model, num_classes, class_weights):
        super(LightningModule, self).__init__()
        self.model = model
        self.num_classes = num_classes
        self.test_step_outputs = []
        self.train_step_outputs = []
        self.val_step_outputs = []  

        # Weighted cross entropy
        weights_tensor = torch.tensor([class_weights[i] for i in range(len(class_weights))], dtype=torch.float)
        self.criterion = SoftCrossEntropyLoss(weights=weights_tensor)
            
    def forward(self, x):
        return self.model(x)
    
    def calculate_and_log_metrics(self, y_true, y_pred, stage):

        metrics = compute_metrics(y_true, y_pred)

        self.log(f'{stage}_Overall_Accuracy', metrics['overall_accuracy'], on_epoch=True, prog_bar=True)
        self.log(f'{stage}_Macro_Precision', metrics['macro_precision'], on_epoch=True, prog_bar=True)
        self.log(f'{stage}_Macro_Recall', metrics['macro_recall'], on_epoch=True, prog_bar=True)
        self.log(f'{stage}_f1_macro', metrics['f1_macro'], on_epoch=True, prog_bar=True)



    def visualize_mri(self, mri_array, slice_index=None):
        """
        Visualize axial, sagittal, and coronal slices of a 3D MRI array.
    
        Args:
            mri_array (torch.Tensor or numpy.ndarray): A 3D MRI array to visualize.
            slice_index (tuple of int or None): The indices of the slices to visualize. 
                                                If None, defaults to the middle slices.
        """
        # If the input is a PyTorch tensor, convert it to a NumPy array
        if isinstance(mri_array, torch.Tensor):
            mri_array = mri_array.cpu().numpy()
    
        # Remove channel dimension if present
        if mri_array.ndim == 4 and mri_array.shape[0] == 1:
            mri_array = mri_array.squeeze(0)
    
        assert mri_array.ndim == 3, "Input array must be 3D."
    
        z, y, x = mri_array.shape
    
        # Default to middle slices if no slice index is provided
        if slice_index is None:
            slice_index = (z // 2, y // 2, x // 2)
    
        # Extract slices
        axial_slice = mri_array[slice_index[0], :, :]
        sagittal_slice = mri_array[:, slice_index[1], :]
        coronal_slice = mri_array[:, :, slice_index[2]]
    
        # Plot slices
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(axial_slice, cmap='gray')
        axes[0].set_title('Axial View')
        axes[1].imshow(sagittal_slice, cmap='gray')
        axes[1].set_title('Sagittal View')
        axes[2].imshow(coronal_slice, cmap='gray')
        axes[2].set_title('Coronal View')
    
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    def random_thresholds(self, min_val, max_val, min_fraction=0.1):
        """
        Generate thresholds for segmentation with guaranteed minimum region sizes.
        
        Args:
        - min_val (float): Minimum value of the distance transform.
        - max_val (float): Maximum value of the distance transform.
        - min_fraction (float): Minimum fraction of the range for each region.
        
        Returns:
        - threshold1 (float): First threshold.
        - threshold2 (float): Second threshold.
        """
        # Convert to CPU and scalar if needed
        if torch.is_tensor(min_val):
            min_val = min_val.cpu().item()
        if torch.is_tensor(max_val):
            max_val = max_val.cpu().item()
        
        # Range of the distance transform
        range_val = max_val - min_val
        min_region_size = range_val * min_fraction
    
        # Ensure thresholds have enough range for each region
        threshold1 = min_val + min_region_size
        threshold2 = max_val - min_region_size
    
        # Introduce randomness within constraints
        threshold1 = np.random.uniform(threshold1, (min_val + max_val) / 2)
        threshold2 = np.random.uniform((min_val + max_val) / 2, threshold2)
    
        return threshold1, threshold2

    
    def apply_distance_transform_mixup(self, x, distance_transforms, y):
        """
        Apply distance transform-based mixup with alpha and lambda parameters.
        Ensures that for a batch size of 2, the first image is mixed with the second and vice versa.
        """
        batch_size = x.size(0)
        assert batch_size == 2, "This implementation is specific to a batch size of 2."
                        
        # Explicit index for swapping (specific to batch size of 2)
        #index = torch.tensor([1, 0]).to(x.device)  # First swaps with second, second swaps with first
        index = torch.randperm(batch_size).to(x.device)

        
        # Extract samples and distance transforms
        x_a, x_b = x, x[index, :]
        dist_a, dist_b = distance_transforms, distance_transforms[index, :]
        y_a, y_b = y, y[index]
        
        # Generate random thresholds for distance-based segmentation
        threshold1_a, threshold2_a = self.random_thresholds(dist_a.min(), dist_a.max(), min_fraction=0.1)
        threshold1_b, threshold2_b = self.random_thresholds(dist_b.min(), dist_b.max(), min_fraction=0.1)

        # Compute regions
        # Compute regions sequentially to ensure non-overlapping
        region1_a = (dist_a <= threshold1_a).float()
        region2_b = ((dist_b > threshold1_a) & (dist_b <= threshold2_b) & (region1_a == 0)).float()
        region3_a = ((dist_a > threshold1_a) & (dist_a <= threshold2_a) & (region1_a == 0) & (region2_b == 0)).float()
        region4_b = ((dist_b > threshold2_b) & (region1_a == 0) & (region2_b == 0) & (region3_a == 0)).float()

        region1_a = region1_a.unsqueeze(1)
        region2_b = region2_b.unsqueeze(1)
        region3_a = region3_a.unsqueeze(1)
        region4_b = region4_b.unsqueeze(1)


        mixed_x = (
            (region1_a * x_a) +
            (region2_b * x_b) +
            (region3_a * x_a) +
            (region4_b * x_b)
        )

        #print(f"Image A - Min: {dist_a.min()}, Max: {dist_a.max()}, Thresholds: {threshold1_a}, {threshold2_a}")
        #print(f"Image B - Min: {dist_b.min()}, Max: {dist_b.max()}, Thresholds: {threshold1_b}, {threshold2_b}")
        #print("[Visualization] Region 1 (A)")
        #self.visualize_mri(region1_a[0] * x_a[0])  # Region 1 from Image A
        #print("[Visualization] Region 2 (B)")
        #self.visualize_mri(region2_b[0] * x_b[0])  # Region 2 from Image B
        #print("[Visualization] Region 3 (A)")
        #self.visualize_mri(region3_a[0] * x_a[0])  # Region 3 from Image A
        #print("[Visualization] Region 4 (B)")
        #self.visualize_mri(region4_b[0] * x_b[0])  # Region 4 from Image B


        #Visualize original images
        #print("[Visualization] Original Image A")
        #self.visualize_mri(x_a[0])  # Visualize the first sample in the batch
        #print("[Visualization] Original Image B")
        #self.visualize_mri(x_b[0])  # Visualize the second sample in the batch
        
        # Visualize mixed image
        #print("[Visualization] Mixed Image")
        #self.visualize_mri(mixed_x[0])  # Visualize the first mixed image
        #self.visualize_mri(mixed_x[1])
        
        #print(f"[apply_distance_transform_mixup] mixed_x shape: {mixed_x.shape}")
        
        # Compute contribution proportions
        pixels_a = (region1_a + region3_a).sum()
        pixels_b = (region2_b + region4_b).sum()
        total_pixels = pixels_a + pixels_b
        
        # Proportional mixing of labels
        prop_a = pixels_a / total_pixels
        prop_b = pixels_b / total_pixels
        
        # Mixed labels
        mixed_y = prop_a * y_a + prop_b * y_b
                
        #print(f"[apply_distance_transform_mixup] mixed_y shape: {mixed_y.shape}, mixed_y: {mixed_y}")
        
        return mixed_x, mixed_y
    
        
    def training_step(self, batch, batch_idx):
        """
        Training step for one batch with debug prints.
        Args:
        - batch: Tuple containing input images (x), labels (y), and distance transforms.
        - batch_idx: Index of the batch.
        """
        
        # Unpack batch
        x, y, distance_transforms = batch
        mixed_x, mixed_y= self.apply_distance_transform_mixup(x, distance_transforms, y)
      
        output = self.model(mixed_x)
       
        loss = self.criterion(output, mixed_y)  # mixed_y is in soft label format
      
        pred_labels = torch.argmax(output, dim=1)      
        y_hard = torch.argmax(y, dim=1)  # Original hard labels for comparison        
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.train_step_outputs.append({'y_true': y_hard, 'y_pred': pred_labels})
        
        return loss



    def on_train_epoch_end(self):

        y_true = torch.cat([x['y_true'] for x in self.train_step_outputs])
        y_pred = torch.cat([x['y_pred'] for x in self.train_step_outputs])

        self.calculate_and_log_metrics(y_true, y_pred, 'train')
        
        self.train_step_outputs = []

    def validation_step(self, batch, batch_idx):
        """
        Validation step for one batch with debug prints.
        Args:
        - batch: Tuple containing input images (x) and labels (y).
        - batch_idx: Index of the batch.
        """
        
        # Unpack batch
        x, y = batch
    
        # Forward pass through the model
        output = self.model(x)
        y_true = torch.argmax(y, dim=1)

        val_loss = self.criterion(output, y)

        pred_labels = torch.argmax(output, dim=1)
    
        self.log('val_loss', val_loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.val_step_outputs.append({'y_true': y_true, 'y_pred': pred_labels})
        
        return val_loss


    def on_validation_epoch_end(self):
        y_true = torch.cat([x['y_true'] for x in self.val_step_outputs])
        y_pred = torch.cat([x['y_pred'] for x in self.val_step_outputs])
    
        self.calculate_and_log_metrics(y_true, y_pred, 'val')
        
        self.val_step_outputs = []

    def test_step(self, batch, batch_idx):
        x, y = batch

        with torch.no_grad():
            output = self.model(x)
            y_indices = torch.argmax(y, dim=1)
            pred_labels = torch.argmax(output, dim=1)

            self.test_step_outputs.append({'y_true': y_indices, 'y_pred': pred_labels})

    def on_test_epoch_end(self):
        # Concatenate all the predictions and true labels from the test step outputs
        y = torch.cat([x['y_true'] for x in self.test_step_outputs])
        output_pred = torch.cat([x['y_pred'] for x in self.test_step_outputs])

        self.calculate_and_log_metrics(y, output_pred, 'test')

        # Calculate Cohen's kappa
        kappa = tmf.cohen_kappa(output_pred, y, task='multiclass', num_classes=self.num_classes)

        # Convert to numpy arrays for metric calculations
        y_np = y.cpu().numpy()
        output_pred_np = output_pred.cpu().numpy()
    
        # Calculate confusion matrix
        confusion_mat = confusion_matrix(y_np, output_pred_np, labels=[0, 1, 2])
        
        # Calculate MCC for each class
        num_classes = confusion_mat.shape[0]
        MCC = np.zeros(num_classes)
        for i in range(num_classes):
            TP = confusion_mat[i, i]
            FP = confusion_mat[:, i].sum() - TP
            FN = confusion_mat[i, :].sum() - TP
            TN = confusion_mat.sum() - (TP + FP + FN)
            
            MCC[i] = (TP * TN - FP * FN) / (np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)) + 1e-9)
        
        # Calculate MACRO_MCC and WEIGHTED_MCC
        MACRO_MCC = MCC.mean()
        WEIGHTED_MCC = (MCC * confusion_mat.sum(axis=0)).sum() / confusion_mat.sum()
        
        # Log Cohen's kappa value
        self.log('test_kappa', kappa, prog_bar=True, sync_dist=True)
        self.log('test_macro_mcc', MACRO_MCC, prog_bar=True, sync_dist=True)
        self.log('test_weighted_mcc', WEIGHTED_MCC, prog_bar=True, sync_dist=True)
        
        # Plot confusion matrix
        df_cm = pd.DataFrame(confusion_mat, index=['NC', 'MCI', 'AD'], columns=['NC', 'MCI', 'AD'])
        plt.figure(figsize=(10, 7))
        sns.heatmap(df_cm, annot=True, cmap='Spectral', fmt='g')
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted labels')
        plt.ylabel('True labels')
        plt.show()
        
        # Clear the outputs for the next epoch
        self.test_step_outputs = []
    
    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0005, nesterov=False)
        exp_lambda: float = 0.95
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: exp_lambda ** epoch)

        return [optimizer], [scheduler]

# Load pre-trained weights
# -- Remove module keyword from state dict since it pre-trained model was trained on dataparallel
weight_dir = '/home/zobia/MedicalNet/pretrain/Genesis_Chest_CT.pt'
checkpoint = torch.load(weight_dir, map_location=torch.device('cpu'))
state_dict = checkpoint['state_dict']
unParalled_state_dict = {}
for key in state_dict.keys():
    unParalled_state_dict[key.replace("module.", "")] = state_dict[key]
base_model.load_state_dict(unParalled_state_dict)

# Initialize target model after loading weights
# -- Target model is the classification model
# -- the base model is originally for segmentation that checkpoint is trained on
num_classes = 3
target_model = TargetNet(base_model, num_classes)

In [ ]:
wandb_logger = WandbLogger(project='Resnet3D', log_model='all', save_dir='./logs')
data_module = DataModule(train_dataset, val_dataset, test_dataset, batch_size, distributed_sampler=False)

# Instantiate PyTorch Lightning Model
#lightning_model = LightningModule(target_model, criterion,num_classes, class_weights=class_weights)
lightning_model = LightningModule(model=target_model, num_classes=num_classes, class_weights=class_weights)

# Best model Checkpoint
checkpoint_callback = ModelCheckpoint(save_top_k=1, dirpath='./ResnetCheck/', 
                                  filename='Best_checkpoint-{epoch:02d}-{val_loss:.3f}',
                                      monitor='val_loss', mode='min')

 # Define early stopping callback
early_stopping_callback = EarlyStopping(
    monitor='val_loss',  # Metric to monitor
    patience=15,          # Number of epochs with no improvement after which training will be stopped
    mode='min'            # Whether to minimize or maximize the monitored metric ('min' for loss, 'max' for accuracy, etc.)
    )

# PyTorch Lightning Trainer
trainer = pl.Trainer(
    logger=wandb_logger,
    max_epochs=90,
    accelerator='gpu',  # 'ddp' accelerator is used for distributed data-parallel training
    precision='16-mixed' ,         # Enables mixed precision training (float16)
    #gradient_clip_val=1.0,
    num_nodes=1,
    devices=1,  # Set the number of GPUs
    #strategy='ddp_notebook',
    accumulate_grad_batches=8,
    #limit_train_batches= 200,
    callbacks=[
            checkpoint_callback,
            early_stopping_callback
            #GradientAccumulationScheduler(batch_size=[4, 8, 16])  # Optional: vary batch size dynamically
        ]
)

# Train the model
#trainer.fit(lightning_model, datamodule=data_module)
   
# Test the model
#best_checkpoint_path = checkpoint_callback.best_model_path
#print(f"Using checkpoint for testing: {best_checkpoint_path}")
#trainer.save_checkpoint(best_checkpoint_path)
#trainer.test(ckpt_path=best_checkpoint_path, datamodule=data_module)

In [ ]:

checkpoint_path ='/home/zobia/ResnetCheck/Best_checkpoint-epoch=76-val_loss=0.759.ckpt'
lightning_model = LightningModule.load_from_checkpoint(checkpoint_path, model=target_model, num_classes=num_classes,class_weights=class_weights)
trainer.test(lightning_model, datamodule=data_module)
